<a href="https://colab.research.google.com/github/svyatoslavna/nlp_hw/blob/main/fine_tuning_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [6]:
!pip install transformers datasets evaluate accelerate gradio -q
!pip install huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


In [7]:
# импорты

import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate
from transformers import pipeline
import gradio as gr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")

In [13]:
# загрузка датасета
ds = load_dataset("fancyzhx/ag_news")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [14]:
# проверка деления на train и test
print(f'Train: {len(ds['train'])}, Test: {len(ds['test'])}')

Train: 120000, Test: 7600


In [ ]:
# загружаем модель
model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# смотрим, какая длина у большинства текстов, чтобы подобрать max_length

lengths = [len(tokenizer(text)["input_ids"]) for text in ds["train"]["text"]]
print(f"Средняя длина: {np.mean(lengths):.0f}")
print(f"95 персентиль: {np.percentile(lengths, 95)}")

Средняя длина: 53
95 персентиль: 82.0


In [17]:
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = ds.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(5000))

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [11]:
# настройка обучения

training_args = TrainingArguments(
    output_dir="./ag_news_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)

In [19]:
# метрики

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [ ]:
# обучение

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.199765,0.181139,0.942800
2,0.128128,0.198570,0.944800
3,0.083597,0.244438,0.945200


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=22500, training_loss=0.14683143547905816, metrics={'train_runtime': 6165.587, 'train_samples_per_second': 58.389, 'train_steps_per_second': 3.649, 'total_flos': 1.6999491211228032e+16, 'train_loss': 0.14683143547905816, 'epoch': 3.0})

In [ ]:
# сохраняю модель, чтобы доделать работу в другой раз

model.save_pretrained('./my_model_2')
tokenizer.save_pretrained('./my_model_2')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_model_2/tokenizer_config.json', './my_model_2/tokenizer.json')

In [3]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r ./my_model_2 '/content/drive/MyDrive/' # сохраняю на Google Disk

Mounted at /content/drive


In [8]:
# вовзращаюсь к работе: выгружаю модель + код в некоторых ячейках сверху заново прогоняю

model = AutoModelForSequenceClassification.from_pretrained('/content/drive/MyDrive/my_model_2')
tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/my_model_2')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [21]:
# оцениваем модель на тестовой выборке

eval_results = trainer.evaluate()
print(f"\nEvaluation results: {eval_results}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.244438,0,0.945200



Evaluation results: {'eval_loss': 0.24443811178207397, 'eval_accuracy': 0.9452}


In [28]:
# делаем маппинг для лейблов, чтобы сразу было ясно, к какой теме отнесена новость моделью

id2label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

model.config.id2label = id2label

In [31]:
# инференс через pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

test_texts = [
    "Boeing to start 737 Max production on new assembly line July 6, CEO says",
    "'I want to thank myself.'That was the phrase emblazoned on Mirra Andreeva's jacket as she lifted her first Grand Slam trophy at the French Open.",
    "At least 15 killed as powerful earthquake strikes southern Philippines",
    "Are we getting to the point where it's safe to gene-edit babies? A team in the US has reported promising results after using an improved form of CRISPR to gene-edit human embryos, but a major issue remains unsolved"
]

for text in test_texts:
    result = classifier(text)[0]
    print(f"Text: {text}\nTopic: {result['label']}, Score: {result['score']:.4f}\n")

Text: Boeing to start 737 Max production on new assembly line July 6, CEO says
Topic: Business, Score: 0.9993

Text: 'I want to thank myself.'That was the phrase emblazoned on Mirra Andreeva's jacket as she lifted her first Grand Slam trophy at the French Open.
Topic: Sports, Score: 0.9974

Text: At least 15 killed as powerful earthquake strikes southern Philippines
Topic: World, Score: 0.9998

Text: Are we getting to the point where it's safe to gene-edit babies? A team in the US has reported promising results after using an improved form of CRISPR to gene-edit human embryos, but a major issue remains unsolved
Topic: Sci/Tech, Score: 0.9969



In [32]:
# инференс с интерфейсом

def predict_topic(text):
    result = classifier(text)[0]
    return {result['label']: result['score']}

demo = gr.Interface(
    fn=predict_topic,
    inputs=gr.Textbox(label="Put here a news article"),
    outputs=gr.Label(label="Topic"),
    title="News classification",
    examples = [
        "Boeing to start 737 Max production on new assembly line July 6, CEO says",
        "'I want to thank myself.'That was the phrase emblazoned on Mirra Andreeva's jacket as she lifted her first Grand Slam trophy at the French Open.",
        "At least 15 killed as powerful earthquake strikes southern Philippines",
        "Are we getting to the point where it's safe to gene-edit babies? A team in the US has reported promising results after using an improved form of CRISPR to gene-edit human embryos, but a major issue remains unsolved"
]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e5e4c10df86dfa2d01.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## **Итог**

Мы выполнили fine-tuning модели `google-bert/bert-base-uncased` на датасете `fancyzhx/ag_news` для задачи классификации новостей. **Accuracy на тестовой выборке достигла 94%**, что является отличным результатом. Инференс на примерах новостей всех четырёх классов - World, Sports, Business, Sci/Tech - был выполнен корректно.